# lecture des données datagouv
étape 0 du workflow de catalogage des données de mobilité data.gouv.fr
2 possibilités 
1) utiliser l'API datagouv-client en python, pour accéder à des objets python ou du json brut, 
permet de récupérer notamment toutes les ressources associées à un dataset : intéressant pour récuperer le titre et la descr des ressources, qui peut aider à bien tagguer les datasets
2) télécharger le catalogue des datasets (plus simple, mais pas d'info sur les ressources des datasets) https://www.data.gouv.fr/datasets/catalogue-des-donnees-de-data-gouv-fr


In [1]:
!pip install datagouv-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.1/813.1 kB 63.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.4 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.7 MB/s  0:00:00


   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/6 [jh2]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 3/6 [urllib3-future]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 3/6 [urllib3-future]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 3/6 [urllib3-future]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [niquests]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [datagouv-client]


In [2]:
from datagouv import Client

In [3]:
import pandas as pd

import requests
import pandas as pd
import time

DATAGOUV_API = "https://www.data.gouv.fr/api/1"

In [4]:
# test de l'API (v1) en brut sans utiliser la lib client
session = requests.Session()
resp = session.get(
            f"{DATAGOUV_API}/datasets/",
            params={"page_size": 10, "page": 1},
            timeout=60,
        )
resp.raise_for_status()
data = resp.json()
items = data.get("data", [])

In [5]:
r=(items[0].get("resources") or [])

In [6]:
## ";".join([f"{i+1}. {d['title']} - {d['description']}" for i, d in enumerate(r or [])])

In [9]:
## affichage d'un dataset avec toutes ses ressources en json à la ligne suivante
#items[0]

In [11]:
# méthode de téléchargement de toutes les 
# une alternative est de télécharger les datasets publiés chaque jour en csv sur https://www.data.gouv.fr/datasets/catalogue-des-donnees-de-data-gouv-fr ( mais il n'y a pas les ressources)

def fetch_datasets_df(page_size=1000, max_pages=None, sleep=0.3) -> pd.DataFrame:
    records = []
    page = 1
    session = requests.Session()

    while True:
        resp = session.get(
            f"{DATAGOUV_API}/datasets/",
            params={"page_size": page_size, "page": page},
            timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()
        items = data.get("data", [])
        if not items:
            break

        for ds in items:
            records.append({
                "id": ds["id"],
                "title": ds.get("title", ""),
                "description": (ds.get("description") or "")[:1000],
                "org": (ds.get("organization") or {}).get("name", ""),
                "org_id": (ds.get("organization") or {}).get("id", ""),
                "tags": ds.get("tags") or [],
                "views": (ds.get("metrics") or {}).get("views", 0),
                "ressources_nb": len(ds.get("resources") or []),
                "ressources_texte": ";".join([f"{i+1}. {d['title']} - {d['description']}" for i, d in enumerate(ds.get("resources") or [])]),
                "downloads": (ds.get("metrics") or {}).get("resources_downloads", 0),
                "license": ds.get("license"),
                "frequency": ds.get("frequency"),
                "last_modified": ds.get("last_modified"),
            })

        print(f"Page {page}: +{len(items)} ({len(records)} total)")
        if not data.get("next_page") or (max_pages and page >= max_pages):
            break
        page += 1
        time.sleep(sleep)

    return pd.DataFrame(records)

In [15]:
import datetime
datetime.datetime.now()
df = fetch_datasets_df()
datetime.datetime.now()
# ça met quelques minutes pour récupérer tous les datasets

Page 1: +1000 (1000 total)


Page 2: +1000 (2000 total)


Page 3: +1000 (3000 total)


Page 4: +1000 (4000 total)


Page 5: +1000 (5000 total)


...

Page 71: +1000 (71000 total)


Page 72: +1000 (72000 total)


Page 73: +1000 (73000 total)


Page 74: +992 (73992 total)


datetime.datetime(2026, 8, 21, 14, 5, 19, 65292)

In [16]:
df.head()

,id,title,description,org,org_id,tags,views,ressources_nb,ressources_texte,downloads,license,frequency,last_modified
0,6a884f10fa89847e4bd683cf,Stock et flux des titres de séjour en France s...,Le jeu de données publié porte sur les statist...,Ministère de l'intérieur,534fff91a3a7292c64a77f53,"[carte-de-sejour, chiffres-cles-de-limmigratio...",0,9,1. 2024_SEJOUR_STOCK_region-monde_duree.csv - ...,0.0,lov2,unknown,2026-08-21T13:14:09.012000+00:00
1,6a8847a57f75b57601210300,Stock et flux des titres de séjour en France s...,Le jeu de données publié porte sur les statist...,Ministère de l'intérieur,534fff91a3a7292c64a77f53,"[carte-de-sejour, chiffres-cles-de-limmigratio...",0,9,1. 2023_SEJOUR_STOCK_region-monde_nationalites...,0.0,lov2,unknown,2026-08-21T12:42:39.648000+00:00
2,6a8847a587f90b83dc8966ed,Populations et territoires exposés au dépassem...,Populations et territoires exposés au dépassem...,Lig'Air,5be949a9634f415deb46af0e,"[dioxyde-dazote-no2, exposition-des-personnes-...",0,8,1. API Geojson - API au format geojson;2. Accè...,0.0,notspecified,severalTimesADay,2026-05-13T17:09:38+00:00
3,6a88479787f90b83dc8966ec,Populations et territoires exposés au dépassem...,Populations et territoires exposés au dépassem...,Lig'Air,5be949a9634f415deb46af0e,"[dioxyde-dazote-no2, exposition-des-personnes-...",0,8,1. API Geojson - API au format geojson;2. Accè...,0.0,notspecified,severalTimesADay,2026-05-13T16:51:34+00:00
4,6a88477e87f90b83dc8966eb,Populations et territoires exposés au dépassem...,Populations et territoires exposés au dépassem...,Lig'Air,5be949a9634f415deb46af0e,"[dioxyde-dazote-no2, exposition-des-personnes-...",0,8,1. API Geojson - API au format geojson;2. Accè...,0.0,notspecified,severalTimesADay,2026-05-13T16:26:44+00:00


In [23]:
df.to_parquet('dsdatagouv.parquet')

In [19]:
import re


In [22]:
df.ressources_texte[3]

'1. API Geojson - API au format geojson;2. Accès au téléchargement des données - ;3. expo_moyan_no2_2024_zas - ;4. expo_moyan_no2_2024_zas - ;5. expo_moyan_pm10_2024_zas - ;6. expo_moyan_pm10_2024_zas - ;7. expo_moyan_pm10_nbrj_2024_zas - ;8. expo_moyan_pm10_nbrj_2024_zas - '